# Kankor Gemini PDF Window Index Inspector

Use this notebook to inspect `data/index/kankor_gemini_pdf_window2`, verify the alignment between the FAISS index and the docstore, and understand how the runtime consumes the stored rows.

What this notebook covers:

- artifact inventory and manifest summary
- metadata integrity and field distribution
- FAISS index structure and vector checks
- a small nearest-neighbor probe over the stored vectors
- the practical role of `metadata.jsonl` in the current RAG stack

In [3]:
from pathlib import Path
import json
from collections import Counter
from pprint import pprint
import statistics

import numpy as np

def resolve_index_dir() -> Path:
    relative = Path("data/index/kankor_gemini_pdf_window2")
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidate = base / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate data/index/kankor_gemini_pdf_window2. "
        "Run the notebook from the repo checkout or build the index first."
    )

DATA_DIR = resolve_index_dir()
INDEX_PATH = DATA_DIR / "index.faiss"
MANIFEST_PATH = DATA_DIR / "manifest.json"
METADATA_PATH = DATA_DIR / "metadata.jsonl"
WINDOW_MANIFEST_PATH = DATA_DIR / "window_manifest.jsonl"
TOC_MANIFEST_PATH = DATA_DIR / "toc_manifest.jsonl"
CLEANUP_REPORT_PATH = DATA_DIR / "retrieval_cleanup_report.json"
AUDIT_PATH = DATA_DIR / "retrieval_cleanup_audit.jsonl"

def load_jsonl(path: Path):
    rows = []
    invalid = []
    with path.open("r", encoding="utf-8") as handle:
        for lineno, raw in enumerate(handle, start=1):
            line = raw.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                invalid.append({
                    "line_number": lineno,
                    "error": str(exc),
                    "preview": line[:140],
                })
    return rows, invalid

def short_text(value, width=180):
    text = " ".join(str(value).split())
    return text if len(text) <= width else text[: width - 1].rstrip() + "…"

def show_record(rows, idx):
    row = rows[idx]
    meta = row.get("metadata") or {}
    return {
        "row": idx,
        "id": row.get("id"),
        "source_id": meta.get("source_id"),
        "page": meta.get("page"),
        "page_range": meta.get("page_range"),
        "subject": meta.get("subject"),
        "grade_band": meta.get("grade_band"),
        "language": meta.get("language"),
        "chapter_number": meta.get("chapter_number"),
        "chapter_heading_kind": meta.get("chapter_heading_kind"),
        "extraction_status": meta.get("extraction_status"),
        "text_preview": short_text(row.get("text", "")),
    }

print(f"Current working dir: {Path.cwd()}")
print(f"Index dir          : {DATA_DIR}")

Current working dir: /home/nasher/Documents/projects/kankor-rag-space/notebooks
Index dir          : /home/nasher/Documents/projects/kankor-rag-space/data/index/kankor_gemini_pdf_window2


In [4]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
metadata_rows, metadata_invalid = load_jsonl(METADATA_PATH)
window_rows, window_invalid = load_jsonl(WINDOW_MANIFEST_PATH)
toc_rows, toc_invalid = load_jsonl(TOC_MANIFEST_PATH) if TOC_MANIFEST_PATH.exists() else ([], [])
cleanup_report = json.loads(CLEANUP_REPORT_PATH.read_text(encoding="utf-8")) if CLEANUP_REPORT_PATH.exists() else {}

print("Files in the index directory:")
for path in sorted(DATA_DIR.iterdir()):
    size_mib = path.stat().st_size / (1024 * 1024)
    print(f"- {path.name:28} {size_mib:7.2f} MiB")

print()
print("Manifest summary:")
pprint(manifest)

print()
print("Row counts:")
pprint({
    "metadata_rows": len(metadata_rows),
    "metadata_invalid_json_lines": len(metadata_invalid),
    "window_rows": len(window_rows),
    "window_invalid_json_lines": len(window_invalid),
    "toc_rows": len(toc_rows),
    "toc_invalid_json_lines": len(toc_invalid),
    "cleanup_alignment": cleanup_report.get("metadata_window_alignment"),
    "cleanup_windows_with_extraction": cleanup_report.get("windows_with_extraction"),
    "cleanup_windows_low_confidence": cleanup_report.get("windows_low_confidence"),
})

print()
print("A few sample rows:")
for idx in [0, 1, 2, len(metadata_rows) - 3, len(metadata_rows) - 2, len(metadata_rows) - 1]:
    pprint(show_record(metadata_rows, idx))


Files in the index directory:
- index.faiss                    45.35 MiB
- manifest.json                   0.00 MiB
- metadata.jsonl                 16.18 MiB
- retrieval_cleanup_audit.jsonl    2.41 MiB
- retrieval_cleanup_report.json    0.00 MiB
- toc_manifest.jsonl              0.03 MiB
- window_manifest.jsonl           0.90 MiB

Manifest summary:
{'chunks': 3870,
 'corpus_version': 'kankor-corpus@2026.03-gemini-pdf-window2',
 'documents': 3870,
 'embedding_backend': 'gemini',
 'embedding_batch_size': 1,
 'embedding_dimension': 3072,
 'embedding_model_id': 'gemini-embedding-2-preview',
 'embedding_task_type': 'RETRIEVAL_DOCUMENT',
 'estimated_pdf_tokens': 1996404,
 'input_dir': 'data/raw_pdfs',
 'metadata_filename': 'metadata.jsonl',
 'pages_total': 7738,
 'pdf_discovered': 43,
 'pdf_failed': 0,
 'pdf_usable': 43,
 'skip_first_pages': 0,
 'window_pages': 2,
 'windows_embedded': 3870,
 'windows_embedded_this_run': 3870,
 'windows_failed': 0,
 'windows_resumed': 0,
 'windows_total': 38

## Why `metadata.jsonl` matters

The runtime does not read `index.faiss` in isolation. `FaissVectorStore.load(...)` loads the FAISS index and then rehydrates the row-aligned document list from the metadata file. Search returns a row index, and that row index is used to look up the corresponding `Document` object.

In practice, `metadata.jsonl` is the docstore for the FAISS index. It is the file that gives each vector:

- the chunk text used for source snippets
- the stable document id
- the metadata used for citations, TOC/topic-locator hints, and local window expansion

So even though the FAISS file holds the vectors, the metadata file is what makes the index usable for grounded answers.

In [5]:
required_fields = [
    "subject_category",
    "subject",
    "grade_band",
    "language",
    "source_type",
    "source_id",
    "page",
]

meta = [row.get("metadata") or {} for row in metadata_rows]
print("Metadata integrity:")
print(f"- invalid JSON lines: {len(metadata_invalid)}")
print(f"- duplicate document ids: {len(metadata_rows) - len({row['id'] for row in metadata_rows})}")
missing_required = []
for i, row in enumerate(metadata_rows):
    row_meta = row.get("metadata") or {}
    for field in required_fields:
        if row_meta.get(field) in (None, ""):
            missing_required.append((i, row.get("id"), field))
print(f"- missing required field samples: {missing_required[:5]}")

print()
for key in [
    "subject_category",
    "subject",
    "grade_band",
    "language",
    "source_type",
    "extraction_status",
    "extraction_source",
    "chapter_heading_kind",
]:
    counter = Counter(str(m.get(key)) for m in meta)
    print(f"{key}:")
    pprint(counter.most_common(12))
    print()

text_lengths = [len(row.get("text", "")) for row in metadata_rows]
print("Text length summary:")
pprint({
    "min": min(text_lengths),
    "median": statistics.median(text_lengths),
    "p90": sorted(text_lengths)[int(len(text_lengths) * 0.9)],
    "max": max(text_lengths),
})

print()
print("TOC rows (first 5):")
for row in toc_rows[:5]:
    pprint(row)

Metadata integrity:
- invalid JSON lines: 0
- duplicate document ids: 0
- missing required field samples: []

subject_category:
[('social_science', 1349),
 ('natural_science', 1204),
 ('languages', 797),
 ('math', 520)]

subject:
[('mathematics', 520),
 ('islamic_studies', 454),
 ('chemistry', 377),
 ('physics', 334),
 ('geography', 324),
 ('pashto', 275),
 ('english', 267),
 ('dari', 255),
 ('history', 247),
 ('biology', 235),
 ('computer_science', 179),
 ('civic_education', 163)]

grade_band:
[('10', 1356), ('11', 1263), ('12', 1251)]

language:
[('fa', 3603), ('ps', 267)]

source_type:
[('explanation', 3870)]

extraction_status:
[('missing', 2292), ('full', 1578)]

extraction_source:
[('window_text', 2292), ('pages.jsonl', 1578)]

chapter_heading_kind:
[('None', 3663),
 ('extraction', 132),
 ('window_text', 51),
 ('window_text_fallback', 24)]

Text length summary:
{'max': 4504, 'median': 2188.0, 'min': 25, 'p90': 3475}

TOC rows (first 5):
{'chapter_number': '1',
 'chapter_title': '

In [6]:
try:
    import faiss
except Exception as exc:
    faiss = None
    print("FAISS is not available in this environment.")
    print("Activate the kankor-rag conda env to inspect index.faiss:")
    print("  conda activate kankor-rag")
    print("  jupyter lab notebooks/inspect_kankor_gemini_pdf_window2.ipynb")
    print(f"Import error: {exc}")
else:
    index = faiss.read_index(str(INDEX_PATH))
    print(type(index).__name__)
    pprint({
        "d": int(index.d),
        "ntotal": int(index.ntotal),
        "is_trained": bool(getattr(index, "is_trained", True)),
    })
    print()
    print("FAISS file header:")
    header = INDEX_PATH.read_bytes()[:16]
    print(header[:4], header[:16].hex())
    print()
    assert index.ntotal == len(metadata_rows) == len(window_rows)
    assert metadata_rows[0]["id"] == window_rows[0]["document_id"]
    assert metadata_rows[-1]["id"] == window_rows[-1]["document_id"]
    first_vec = index.reconstruct(0)
    last_vec = index.reconstruct(index.ntotal - 1)
    print("Vector norms (should be close to 1.0 because the builder normalizes vectors):")
    pprint({
        "first_norm": float(np.linalg.norm(first_vec)),
        "last_norm": float(np.linalg.norm(last_vec)),
        "first_head": [float(v) for v in first_vec[:8]],
        "last_head": [float(v) for v in last_vec[:8]],
    })

    def neighbors_from_row(row_idx, top_k=5):
        query = index.reconstruct(int(row_idx)).astype(np.float32).reshape(1, -1)
        faiss.normalize_L2(query)
        scores, ids = index.search(query, top_k)
        results = []
        for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), start=1):
            row = metadata_rows[int(idx)]
            meta = row.get("metadata") or {}
            results.append({
                "rank": rank,
                "score": round(float(score), 6),
                "row": int(idx),
                "id": row.get("id"),
                "source_id": meta.get("source_id"),
                "page": meta.get("page"),
                "page_range": meta.get("page_range"),
                "chapter_number": meta.get("chapter_number"),
                "chapter_heading_kind": meta.get("chapter_heading_kind"),
                "title": meta.get("title"),
                "snippet": short_text(row.get("text", ""), 180),
            })
        return results

    print()
    print("Nearest neighbors for a few sample rows:")
    for sample_row in [2, 1000, len(metadata_rows) - 1]:
        print(f"\nSample row {sample_row}: {metadata_rows[sample_row]['id']}")
        pprint(neighbors_from_row(sample_row, top_k=5))

IndexFlatIP
{'d': 3072, 'is_trained': True, 'ntotal': 3870}

FAISS file header:
b'IxFI' 49784649000c00001e0f000000000000

Vector norms (should be close to 1.0 because the builder normalizes vectors):
{'first_head': [-0.005074072163552046,
                0.011239627376198769,
                0.005858199205249548,
                -0.008116566576063633,
                0.0050679780542850494,
                -0.012035615742206573,
                0.007919617928564548,
                0.0005392564344219863],
 'first_norm': 0.9999992847442627,
 'last_head': [0.011379211209714413,
               0.005741569679230452,
               -0.017110511660575867,
               -0.003641938092187047,
               0.00665612006559968,
               0.004815832246094942,
               -0.008264322765171528,
               -0.00037658392102457583],
 'last_norm': 1.0000001192092896}

Nearest neighbors for a few sample rows:

Sample row 2: 6d471835d4c19c12
[{'chapter_heading_kind': 'extraction',
  'ch

## What we learn from this corpus

- `index.faiss` is a flat inner-product FAISS index with 3,072-dimensional vectors.
- The index has 3,870 rows and the metadata file has 3,870 aligned JSONL records.
- `metadata.jsonl` is not just a log file. It is the docstore the runtime uses to recover text, ids, page numbers, and retrieval metadata for each vector.
- The file is syntactically valid JSONL in this checkout, but the corpus quality is mixed: many rows are `extraction_status=missing`, so the text often falls back to window-level OCR/extraction recovery.
- The current runtime already expects this row-aligned docstore. If you want to simplify it later, you need a replacement file with the same row order and the fields used by citations, topic-locator, and local expansion.